# Zero-Shot Semantic Matching with GTE: Automatic Course Classification by Career Path

**Objective**: Build an algorithm to automatically classify courses into career paths of 5 Departments and assign weights based on GTE embeddings.

**Author**: IT2041.CH201 Course Generation System

---

## Install Libraries

In [ ]:
!pip install sentence-transformers scikit-learn pandas numpy

In [ ]:
import json
import re
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Set, Tuple, Optional
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

print("Libraries imported successfully")

## Load Data

In [ ]:
# Change to parent directory if running from notebooks folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

ROOT = Path(".").resolve()
print(f"Root directory: {ROOT}")

# Load course catalog
catalog_file = ROOT / "data" / "raw" / "course_catalog.csv"
print(f"Loading catalog from: {catalog_file}")
df_catalog = pd.read_csv(catalog_file)

# Merge with descriptions from daa.uit.edu.vn (if available)
desc_file = ROOT / "data" / "raw" / "course_descriptions.json"
if desc_file.exists():
    with open(desc_file, 'r', encoding='utf-8') as f:
        descriptions = json.load(f)
    df_desc = pd.DataFrame(descriptions)
    # Merge by course_id
    df_catalog = df_catalog.merge(df_desc[['course_id', 'description']], on='course_id', how='left')
    print(f"Loaded {len(descriptions)} course descriptions")
else:
    df_catalog['description'] = ''
    print(f"No course_descriptions.json found")

print(f"Total courses: {len(df_catalog)}")
print(f"\nFirst 5 courses:")
print(df_catalog[['course_id', 'name_vi', 'managed_by']].head())

## Class 1: GTESemanticMatcher

**Task**: Calculate % compatibility between course and each Anchor Profile (career path).

In [ ]:
class GTESemanticMatcher:
    """
    Zero-Shot Semantic Matching with Anchor Profiles.
    """
    
    def __init__(self, model_name: str = 'thenlper/gte-small'):
        print(f"Loading GTE model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        
        # Load Anchor Profiles
        anchor_file = Path("notebooks") / "anchor_profiles.json"
        if not anchor_file.exists():
            anchor_file = Path(".") / "notebooks" / "anchor_profiles.json"
        
        print(f"Loading anchor profiles from: {anchor_file}")
        with open(anchor_file, 'r', encoding='utf-8') as f:
            self.anchor_profiles = json.load(f)
        
        self.course_embeddings = None
        self.anchor_embeddings = {}
        self.course_ids = None
        self.course_descriptions = None
        self.department_map = {}
        
        # TF-IDF vectorizer
        self.tfidf_vectorizer = TfidfVectorizer(max_features=500, stop_words=['of', 'and', 'the', 'a', 'in', 'to', 'for', 'with', 'from', 'by'])
        self.tfidf_matrix = None
        self.anchor_tfidf = {}
        
        print(f"GTE model loaded successfully")
        print(f"Number of departments: {len(self.anchor_profiles)}")
        for dept, info in self.anchor_profiles.items():
            print(f"  - {dept}: {len(info['anchors'])} career paths")
    
    def _build_course_description(self, row: pd.Series) -> str:
        """
        Build rich course description.
        """
        parts = []
        
        # Description from web
        desc = str(row.get('description', ''))
        if desc and desc.lower() not in ('nan', '', 'no description'):
            parts.append(desc)
        
        # Course name (Vietnamese)
        name_vi = str(row.get('name_vi', ''))
        if name_vi and name_vi.lower() != 'nan':
            parts.append(name_vi)
        
        # Course name (English)
        name_en = str(row.get('name_en', ''))
        if name_en and name_en.lower() != 'nan':
            parts.append(name_en)
        
        # Course ID prefix
        course_id = str(row.get('course_id', ''))
        if course_id:
            prefix = re.match(r'([A-Za-z]+)', course_id)
            if prefix:
                prefix_map = {
                    'IT': 'information technology programming',
                    'CS': 'computer science artificial intelligence',
                    'CE': 'computer engineering hardware embedded',
                    'SE': 'software engineering',
                    'IS': 'information systems business',
                    'NT': 'network communication',
                    'DS': 'data science',
                    'AI': 'artificial intelligence',
                    'EC': 'e-commerce',
                }
                if prefix.group(1) in prefix_map:
                    parts.append(prefix_map[prefix.group(1)])
        
        return ' '.join(parts)
    
    def fit(self, df_catalog: pd.DataFrame):
        """
        Encode entire course list and Anchor Profiles.
        """
        print(f"\nStep 1: Building course descriptions...")
        descriptions = []
        dept_map = {}
        for _, row in df_catalog.iterrows():
            desc = self._build_course_description(row)
            descriptions.append(desc)
            cid = str(row.get('course_id', ''))
            dept = str(row.get('managed_by', '')).strip()
            dept_map[cid] = dept
        
        self.course_ids = df_catalog['course_id'].tolist()
        self.course_descriptions = descriptions
        self.department_map = dept_map
        
        print(f"Built descriptions for {len(descriptions)} courses")
        print(f"\nStep 2: GTE Encoding...")
        self.course_embeddings = self.model.encode(descriptions, show_progress_bar=True)
        print(f"Shape: {self.course_embeddings.shape}")
        
        print(f"\nStep 3: TF-IDF Encoding...")
        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(descriptions).toarray()
        print(f"TF-IDF shape: {self.tfidf_matrix.shape}")
        
        print(f"\nStep 4: Encoding Anchor Profiles...")
        for dept_code, dept_info in self.anchor_profiles.items():
            self.anchor_embeddings[dept_code] = {}
            self.anchor_tfidf[dept_code] = {}
            
            for anchor_key, anchor_info in dept_info['anchors'].items():
                anchor_text = anchor_info['description'] + ' ' + ' '.join(anchor_info['keywords'])
                anchor_emb = self.model.encode([anchor_text])[0]
                self.anchor_embeddings[dept_code][anchor_key] = anchor_emb
                anchor_tfidf = self.tfidf_vectorizer.transform([anchor_text]).toarray()[0]
                self.anchor_tfidf[dept_code][anchor_key] = anchor_tfidf
            
            print(f"  {dept_code}: {len(dept_info['anchors'])} anchors")
        
        print(f"\nEncoding complete!")
        return self
    
    def compute_similarity(self, course_idx: int, dept_code: str, anchor_key: str,
                           alpha: float = 0.7) -> float:
        """
        Calculate Cosine Similarity between course and anchor profile.
        """
        course_vec = self.course_embeddings[course_idx].reshape(1, -1)
        anchor_vec = self.anchor_embeddings[dept_code][anchor_key].reshape(1, -1)
        gte_sim = cosine_similarity(course_vec, anchor_vec)[0][0]
        
        course_tfidf = self.tfidf_matrix[course_idx].reshape(1, -1)
        anchor_tfidf = self.anchor_tfidf[dept_code][anchor_key].reshape(1, -1)
        tfidf_sim = cosine_similarity(course_tfidf, anchor_tfidf)[0][0]
        
        combined = alpha * gte_sim + (1 - alpha) * tfidf_sim
        return combined
    
    def match_course_to_department(self, course_id: str, dept_code: str) -> Dict:
        """Match a course with all career paths of a Department."""
        if course_id not in self.course_ids:
            return {}
        course_idx = self.course_ids.index(course_id)
        results = {}
        for anchor_key, anchor_info in self.anchor_profiles[dept_code]['anchors'].items():
            sim = self.compute_similarity(course_idx, dept_code, anchor_key)
            percentage = max(0, min(100, sim * 100))
            results[anchor_key] = {
                'name': anchor_info['name'],
                'score': float(sim),
                'percentage': round(percentage, 1),
            }
        results = dict(sorted(results.items(), key=lambda x: x[1]['score'], reverse=True))
        return results
    
    def get_best_anchor(self, course_id: str, dept_code: str) -> Optional[Dict]:
        """Get the best matching career path for a course in a Department."""
        matches = self.match_course_to_department(course_id, dept_code)
        if not matches:
            return None
        best_key = list(matches.keys())[0]
        return {
            'anchor_key': best_key,
            **matches[best_key]
        }

print("GTESemanticMatcher class defined")

## Demo: Test the Matcher

In [ ]:
# Initialize the matcher
print("Initializing GTESemanticMatcher...")
matcher = GTESemanticMatcher()

# Fit on catalog
print("\nFitting matcher on catalog...")
matcher.fit(df_catalog)

print("\n" + "="*60)
print("DEMO: Testing course matching")
print("="*60)

# Test with a few courses
test_courses = ['IT001', 'CS001', 'NT001']
test_dept = 'KHMT'

for course_id in test_courses:
    if course_id in matcher.course_ids:
        print(f"\nCourse: {course_id}")
        course_row = df_catalog[df_catalog['course_id'] == course_id]
        if not course_row.empty:
            print(f"  Name: {course_row.iloc[0]['name_vi']}")
            print(f"  Department: {course_row.iloc[0]['managed_by']}")
        
        matches = matcher.match_course_to_department(course_id, test_dept)
        if matches:
            print(f"  Matches with {test_dept}:")
            for anchor_key, match_info in matches.items():
                print(f"    - {match_info['name']}: {match_info['percentage']:.1f}%")
    else:
        print(f"\nCourse {course_id} not found in catalog")

print("\n" + "="*60)
print("Demo completed successfully!")
print("="*60)

## Simulation: Course Recommendation System

**Objective**: Simulate a student inputting their schedule, faculty, and learned courses to get course recommendations.

In [ ]:
# Simulate student input
print("="*70)
print("COURSE RECOMMENDATION SYSTEM - SIMULATION")
print("="*70)

# 1. Student's Faculty/Department
student_faculty = 'KHMT'  # Khoa Khoa học Máy tính
print(f"\n1. FACULTY: {student_faculty}")
print(f"   Faculty Name: {matcher.anchor_profiles[student_faculty]['name']}")
print(f"   Description: {matcher.anchor_profiles[student_faculty]['description']}")

# 2. Student's Schedule (Thời khoá biểu)
student_schedule = {
    'Monday': ['08:00-10:00', '13:00-15:00'],
    'Tuesday': ['10:00-12:00'],
    'Wednesday': ['08:00-10:00', '15:00-17:00'],
    'Thursday': ['13:00-15:00'],
    'Friday': ['10:00-12:00', '14:00-16:00']
}
print(f"\n2. SCHEDULE (Thời khoá biểu):")
for day, times in student_schedule.items():
    print(f"   {day}: {', '.join(times)}")

# 3. Learned Courses (Các môn đã học)
learned_courses = [
    'IT001',  # Nhập môn lập trình
    'IT003',  # Cấu trúc dữ liệu và giải thuật
    'IT004',  # Cơ sở dữ liệu
    'CS106',  # Trí tuệ nhân tạo
    'CS112',  # Phân tích và thiết kế thuật toán
]
print(f"\n3. LEARNED COURSES (Các môn đã học):")
for course_id in learned_courses:
    if course_id in matcher.course_ids:
        course_row = df_catalog[df_catalog['course_id'] == course_id]
        if not course_row.empty:
            print(f"   - {course_id}: {course_row.iloc[0]['name_vi']}")
    else:
        print(f"   - {course_id}: (Not found in catalog)")

In [ ]:
# Generate recommendations based on learned courses
print("\n" + "="*70)
print("COURSE RECOMMENDATIONS")
print("="*70)

# Get all available courses in the faculty
faculty_courses = df_catalog[df_catalog['managed_by'] == student_faculty]['course_id'].tolist()
print(f"\nTotal courses in {student_faculty}: {len(faculty_courses)}")

# Filter out already learned courses
available_courses = [c for c in faculty_courses if c not in learned_courses]
print(f"Available courses (not yet learned): {len(available_courses)}")

# Calculate recommendation scores for each available course
recommendations = {}

for course_id in available_courses[:20]:  # Limit to first 20 for demo
    if course_id in matcher.course_ids:
        # Get matches with all career paths in the faculty
        career_matches = {}
        for anchor_key in matcher.anchor_profiles[student_faculty]['anchors'].keys():
            matches = matcher.match_course_to_department(course_id, student_faculty)
            if matches and anchor_key in matches:
                career_matches[anchor_key] = matches[anchor_key]['percentage']
        
        # Calculate average score across all career paths
        if career_matches:
            avg_score = sum(career_matches.values()) / len(career_matches)
            course_row = df_catalog[df_catalog['course_id'] == course_id]
            if not course_row.empty:
                recommendations[course_id] = {
                    'name': course_row.iloc[0]['name_vi'],
                    'score': avg_score,
                    'career_matches': career_matches
                }

# Sort by score
sorted_recommendations = sorted(recommendations.items(), key=lambda x: x[1]['score'], reverse=True)

print(f"\nTop 10 Recommended Courses:")
print("-" * 70)
for idx, (course_id, info) in enumerate(sorted_recommendations[:10], 1):
    print(f"\n{idx}. {course_id}: {info['name']}")
    print(f"   Overall Score: {info['score']:.1f}%")
    print(f"   Career Path Alignment:")
    for career, score in sorted(info['career_matches'].items(), key=lambda x: x[1], reverse=True):
        career_name = matcher.anchor_profiles[student_faculty]['anchors'][career]['name']
        print(f"     - {career_name}: {score:.1f}%")

In [ ]:
# Summary and insights
print("\n" + "="*70)
print("RECOMMENDATION SUMMARY")
print("="*70)

print(f"\nStudent Profile:")
print(f"  Faculty: {matcher.anchor_profiles[student_faculty]['name']}")
print(f"  Courses Completed: {len(learned_courses)}")
print(f"  Available Time Slots: {sum(len(times) for times in student_schedule.values())} slots/week")

print(f"\nRecommendation Strategy:")
print(f"  1. Analyzed {len(available_courses)} available courses")
print(f"  2. Matched each course against {len(matcher.anchor_profiles[student_faculty]['anchors'])} career paths")
print(f"  3. Ranked by relevance to student's faculty and learning history")

if sorted_recommendations:
    top_course = sorted_recommendations[0]
    print(f"\nTop Recommendation:")
    print(f"  Course: {top_course[0]} - {top_course[1]['name']}")
    print(f"  Relevance Score: {top_course[1]['score']:.1f}%")
    print(f"  Best Aligned Career Path: {max(top_course[1]['career_matches'].items(), key=lambda x: x[1])[0]}")

print("\n" + "="*70)
print("Simulation completed successfully!")
print("="*70)